# Data Preparation

This file prepares the MAP Supplementary Homicide Report, 2020 LEMAS, and 2019--2023 ACS data used in the homicide clearance analysis. It performs cleaning, feature engineering, and merging of the datasets.

Raw data files are not included in the repository. Place the downloaded source files under the `data/` directory using the folder structure referenced in the setup cell. 

### 1. Load & Setup

In [1]:
import pandas as pd
import numpy as np
import os
import re
import unicodedata

In [ ]:
from pathlib import Path

# expected repository structure:
# notebooks/data_preparation.ipynb
# data/map/, data/lemas/, data/poverty/, data/unemployment/
# data/processed/ is created for cleaned files

base = Path('../data')

map_path = base / "map" / "SHR76_24a.csv"
lemas_path = base / "lemas" / "DS0001" / "38651-0001-Data.tsv"
poverty_path = base / "poverty" / "ACSST5Y2023.S1701-Data.csv"
unemployment_path = base / "unemployment" / "ACSDT5Y2023.B23025-Data.csv"

# for cleaned files
processed_dir = base / "processed"
processed_dir.mkdir(parents = True, exist_ok = True)

In [3]:
print("Loading MAP...")
map_df = pd.read_csv(map_path, low_memory = False)
print(f"MAP Loaded: {map_df.shape[0]} rows, {map_df.shape[1]} columns")

print("Loading LEMAS...")
lemas_df = pd.read_csv(lemas_path, sep = '\t', low_memory = False)
print(f"LEMAS Loaded: {lemas_df.shape[0]} rows, {lemas_df.shape[1]} columns")

print("Loading Poverty Data...")
poverty_df = pd.read_csv(poverty_path, low_memory = False, skiprows = [1])
print(f"Poverty Data Loaded: {poverty_df.shape[0]} rows, {poverty_df.shape[1]} columns")

print("Loading Unemployment Data...")
unemployment_df = pd.read_csv(unemployment_path, low_memory = False, skiprows = [1])
print(f"Unemployment Data Loaded: {unemployment_df.shape[0]} rows, {unemployment_df.shape[1]} columns")


Loading MAP...
MAP Loaded: 913067 rows, 30 columns
Loading LEMAS...
LEMAS Loaded: 3499 rows, 437 columns
Loading Poverty Data...
Poverty Data Loaded: 3222 rows, 375 columns
Loading Unemployment Data...
Unemployment Data Loaded: 3222 rows, 17 columns


### 2. Murder Accountability Project (MAP)

#### 2.1 Initial Exploration

The Murder Accountability Project (MAP) dataset contains homicide records
from 1976–2024. The initial exploration focuses on identifying the relevant
homicide cases, checking the target variable, and assessing unknown values
in key variables.

In [4]:
# filter for homicide cases
print(f"Rows Before Filters: {len(map_df):,}")
map_df = map_df[map_df['Homicide'] == "Murder and non-negligent manslaughter"]

# remove raw codes from circumstance
map_df = map_df[~map_df['Circumstance'].isin(['31', '30'])]

print(f"Rows After Filtering: {len(map_df):,}")

Rows Before Filters: 913,067
Rows After Filtering: 891,358


In [5]:
# create target variable 
map_df['SOLVED'] = map_df['Solved'].map({'Yes': 1, 'No': 0})

print(f"Solved:"
      f"{map_df['SOLVED'].sum():,} ({map_df['SOLVED'].mean():.1%})")

print(f"Unsolved:"
      f"{(map_df['SOLVED']==0).sum():,} ({1-map_df['SOLVED'].mean():.1%})")

Solved:626,367 (70.3%)
Unsolved:264,991 (29.7%)


In [6]:
# check unknown values
print("Unknown Values per Column:")

n = (map_df['VicAge'] == 999).sum()
print(f"VicAge: {n:,} unknown ({n/len(map_df):.1%})")

unknown_codes = {
    'VicSex': 'Unknown',
    'VicRace': 'Unknown', 
    'VicEthnic': 'Unknown or not reported',
    'Relationship': 'Relationship not determined',
    'Circumstance': 'Circumstances undetermined'
}

for col, code in unknown_codes.items():
    if col in map_df.columns:
        n = (map_df[col] == code).sum()
        print(f"{col}: {n:,} ({n/len(map_df):.1%})")

Unknown Values per Column:
VicAge: 13,148 unknown (1.5%)
VicSex: 1,734 (0.2%)
VicRace: 11,067 (1.2%)
VicEthnic: 496,871 (55.7%)
Relationship: 357,439 (40.1%)
Circumstance: 256,413 (28.8%)


#### 2.2 Cleaning

In [7]:
# remove records with unknown ages 
map_df = map_df[map_df['VicAge'] != 999]

The following variables are removed before modelling.

- `VicEthnic` is excluded because of the high proportion of unknown values.
- `Relationship` is excluded because its missingness is strongly associated
  with unsolved cases and may introduce data leakage.
- Offender characteristics are excluded because they are unavailable
  when a case is unsolved.
- Administrative variables that do not provide useful predictive information
  are also removed.

The remaining variables are retained based on their relevance to homicide
clearance and the approach used by Campedelli (2022).

In [8]:
# remove variables that aren't used in the analysis
cols_to_drop = ['Solved', 'ActionType', 'Homicide', 
                'Situation', 'Subcircum', 'FileDate', 
                'MSA', 'Source', 'VicEthnic', 'Relationship',
                'OffAge', 'OffSex', 'OffRace', 'OffEthnic'
]

map_df = map_df.drop(columns = cols_to_drop, errors = 'ignore')
print(f"Columns Remaining: {list(map_df.columns)}")

Columns Remaining: ['ID', 'CNTYFIPS', 'Ori', 'State', 'Agency', 'Agentype', 'Year', 'Month', 'Incident', 'VicAge', 'VicSex', 'VicRace', 'Weapon', 'Circumstance', 'VicCount', 'OffCount', 'SOLVED']


#### 2.3 Feature Engineering

In [9]:
# create five year victim age bands
age_bins = [0, 5, 10, 15, 20, 25, 30, 35, 40, 45, 50, 
            55, 60, 65, 70, 75, 80, 85, 90, 95, 100]

age_labels = [f"{age_bins[i]} - {age_bins[i+1]-1}" for i in range(len(age_bins) - 1)]

map_df['VicAgeBand'] = pd.cut(map_df['VicAge'], 
                              bins = age_bins, 
                              labels = age_labels, 
                              right = False)

print(f"\nVictim Age Distribution:")
print(map_df['VicAgeBand'].value_counts().sort_index())


Victim Age Distribution:
VicAgeBand
0 - 4       26990
5 - 9        6440
10 - 14     10128
15 - 19     93483
20 - 24    153069
25 - 29    138031
30 - 34    111631
35 - 39     86438
40 - 44     66759
45 - 49     50329
50 - 54     39146
55 - 59     29101
60 - 64     21638
65 - 69     15183
70 - 74     11106
75 - 79      8209
80 - 84      5705
85 - 89      3144
90 - 94      1233
95 - 99       447
Name: count, dtype: int64


In [10]:
# collapse tiny circumstance categories into other
circ_counts = map_df['Circumstance'].value_counts()
small_circs = circ_counts[circ_counts < 100].index.tolist()

map_df['Circumstance'] = map_df['Circumstance'].apply(
    lambda x: 'Other' if x in small_circs else x
)

print(f"\nCircumstance Categories Remaining: {map_df['Circumstance'].nunique()}")
print(map_df['Circumstance'].value_counts())


Circumstance Categories Remaining: 28
Circumstance
Circumstances undetermined              249567
Other arguments                         229774
Other                                   116074
Robbery                                  62132
Narcotic drug laws                       32445
Other - not specified                    31496
Juvenile gang killings                   26010
Felon killed by police                   18427
Brawl due to influence of alcohol        16147
Argument over money or property          15927
Felon killed by private citizen          15491
All suspected felony type                13395
Lovers triangle                          12658
Gangland killings                         7085
Burglary                                  6916
Brawl due to influence of narcotics       5169
Arson                                     5077
Rape                                      4237
Motor vehicle theft                       1680
Institutional killings                    1503
Other se

In [11]:
# create agency-year-month identifier
map_df['agency_year_month'] = (
    map_df['Ori'].astype(str) + "_" +
    map_df['Year'].astype(str) + "_" +
    map_df['Month'].astype(str)
)

overlap_counts = map_df.groupby('agency_year_month')['agency_year_month'].transform('count')

map_df['Monthly_Overlap'] = (overlap_counts > 1).astype(int)

overlap_pct = map_df['Monthly_Overlap'].mean()

print(f"Monthly Overlap: {overlap_pct:.1%} of cases have another homicide in the same month for the same agency")

Monthly Overlap: 72.0% of cases have another homicide in the same month for the same agency


In [12]:
# create near lemas flag for sensitivity analysis
map_df['NearLemas'] = map_df['Year'].between(2018, 2022).astype(int)
near_lemas_count = map_df['NearLemas'].sum()

print(f"NearLemas: {near_lemas_count:,} cases from 2018-2022 ({near_lemas_count/len(map_df):.1%})")

NearLemas: 94,938 cases from 2018-2022 (10.8%)


In [13]:
# standardise ori for joining 
map_df['Ori'] = map_df['Ori'].str.strip().str.upper()

#### 2.4 Exploratory Analysis

In [14]:
print(f"Final Shape: {map_df.shape[0]:,} rows, {map_df.shape[1]} columns")
print(f"Year Range: {map_df['Year'].min()} - {map_df['Year'].max()}")
print(f"Unique States: {map_df['State'].nunique()}")
print(f"Unique Agencies: {map_df['Ori'].nunique():,}")

print(f"\nColumns in Cleaned MAP Dataset:")
print(list(map_df.columns))

Final Shape: 878,210 rows, 21 columns
Year Range: 1976 - 2024
Unique States: 51
Unique Agencies: 13,200

Columns in Cleaned MAP Dataset:
['ID', 'CNTYFIPS', 'Ori', 'State', 'Agency', 'Agentype', 'Year', 'Month', 'Incident', 'VicAge', 'VicSex', 'VicRace', 'Weapon', 'Circumstance', 'VicCount', 'OffCount', 'SOLVED', 'VicAgeBand', 'agency_year_month', 'Monthly_Overlap', 'NearLemas']


In [15]:
print("\nSummary Statistics:")
map_df[['VicAge', 'VicCount', 'OffCount', 'Year']].describe().round(2)


Summary Statistics:


,VicAge,VicCount,OffCount,Year
count,878210.00,878210.00,878210.00,878210.00
mean,33.24,0.14,0.20,1999.18
std,16.07,0.61,0.63,14.35
min,0.00,0.00,0.00,1976.00
25%,22.00,0.00,0.00,1987.00
50%,30.00,0.00,0.00,1998.00
75%,42.00,0.00,0.00,2012.00
max,99.00,52.00,40.00,2024.00


In [16]:
missing = map_df.isnull().sum()
missing = missing[missing > 0]
print("Missing Values:" if len(missing) > 0 else "No missing values")
if len(missing) > 0:
    print(missing)

No missing values


In [17]:
print("Clearance Rate by Year (2014+):")
print(map_df[map_df['Year'] >= 2014].groupby('Year')['SOLVED'].mean().round(3))

print("\nClearance Rate by VicRace:")
print(map_df.groupby('VicRace')['SOLVED'].agg(['mean', 'count']).round(3))

print("\nClearance Rate by VicSex:")
print(map_df.groupby('VicSex')['SOLVED'].agg(['mean', 'count']).round(3))

print("\nClearance Rate by Weapon:")
print(map_df.groupby('Weapon')['SOLVED'].mean().sort_values().round(3))

print("\nClearance Rate by Circumstance:")
print(map_df.groupby('Circumstance')['SOLVED'].mean().sort_values().round(3))

Clearance Rate by Year (2014+):
Year
2014    0.688
2015    0.675
2016    0.663
2017    0.671
2018    0.679
2019    0.683
2020    0.676
2021    0.700
2022    0.706
2023    0.721
2024    0.766
Name: SOLVED, dtype: float64

Clearance Rate by VicRace:
                                      mean   count
VicRace                                           
American Indian or Alaskan Native    0.795    7065
Asian                                0.723   12657
Black                                0.661  425936
Native Hawaiian or Pacific Islander  0.816     293
Unknown                              0.666    8574
White                                0.750  423685

Clearance Rate by VicSex:
          mean   count
VicSex                
Female   0.782  194483
Male     0.684  683160
Unknown  0.540     567

Clearance Rate by Weapon:
Weapon
Firearm, type not stated                0.551
Strangulation - hanging                 0.555
Other gun                               0.561
Other or type unknown         

In [18]:
map_out_path = os.path.join(processed_dir, "map_clean.csv")
map_df.to_csv(map_out_path, index = False)
print(f"Saved Cleaned MAP data to {map_out_path}")

Saved Cleaned MAP data to /Users/mariamkhan/Desktop/Thesis/Data/Processed/map_clean.csv


### 3. Law Enforcement Management and Administrative Statistics (LEMAS)

#### 3.1 Initial Exploration
The LEMAS dataset provides agency-level information on law enforcement staffing, resources, budgets, training, and other organisational characteristics. Initial exploration is used to assess the structure of the dataset and identify variables relevant to the analysis.

In [19]:
print(f"Shape: {lemas_df.shape[0]:,} rows, {lemas_df.shape[1]} columns")
print(f"Unique Agencies: {lemas_df['ORI7'].nunique():,}")

# key columns 
lemas_cols = ['ORI7', 'STATE', 'FTSWORN', 'PTSWORN', 'TOTFTEMP', 'OPBUDGET', 
              'DET_SWN', 'PERS_TRN_ACAD', 'PERS_TRN_INSVC', 'PERS_EDU_MIN']

print(f"\nSample of Key Columns:")
lemas_df[lemas_cols].head()

Shape: 3,499 rows, 437 columns
Unique Agencies: 3,290

Sample of Key Columns:


,ORI7,STATE,FTSWORN,PTSWORN,TOTFTEMP,OPBUDGET,DET_SWN,PERS_TRN_ACAD,PERS_TRN_INSVC,PERS_EDU_MIN
0,-9,TX,4060,0,10061,1.210854e+09,747,1418,20,3
1,-1,GA,1042,4,1483,2.033004e+08,0,1033,20,4
2,-1,UT,578,0,753,1.116928e+08,100,760,80,4
3,-9,SC,948,0,1258,1.692746e+08,6,480,40,4
4,AKASP00,AK,274,0,458,1.537847e+08,65,1000,0,4


In [20]:
# missing values 
for col in ['FTSWORN', 'PTSWORN', 'TOTFTEMP', 
            'OPBUDGET', 'DET_SWN', 'PERS_TRN_ACAD',
            'PERS_TRN_INSVC', 'PERS_EDU_MIN']:
        n = lemas_df[col].isin([-9, -8]).sum()
        print(f"{col}: {n} missing codes (-9/-8) ({n/len(lemas_df):.1%})")

FTSWORN: 27 missing codes (-9/-8) (0.8%)
PTSWORN: 801 missing codes (-9/-8) (22.9%)
TOTFTEMP: 784 missing codes (-9/-8) (22.4%)
OPBUDGET: 831 missing codes (-9/-8) (23.7%)
DET_SWN: 0 missing codes (-9/-8) (0.0%)
PERS_TRN_ACAD: 833 missing codes (-9/-8) (23.8%)
PERS_TRN_INSVC: 821 missing codes (-9/-8) (23.5%)
PERS_EDU_MIN: 784 missing codes (-9/-8) (22.4%)


#### 3.2 Cleaning

LEMAS uses negative values such as `-9` and `-8` to represent unavailable or non-applicable responses. We will keep all missing values and check case-level missing rate after joining LEMAS data to MAP data. The agency identifier (`ORI7`) is standardised to ensure it can be matched consistently with the MAP dataset.

In [21]:
lemas_clean = lemas_df[lemas_cols].copy()

# replace lemas missing value codes with nan
for col in lemas_cols[2:]:
    lemas_clean[col] = lemas_clean[col].replace([-9, -8], np.nan)

# standardise or17 for joining
lemas_clean['ORI7'] = lemas_clean['ORI7'].str.strip().str.upper()

# drop ori7 placeholder codes
invalid_ori7 = ['-9', '-8', '-1']

print(f"Rows Before Dropping Invalid ORI7: {len(lemas_clean):,}")
lemas_clean = lemas_clean[~lemas_clean['ORI7'].isin(invalid_ori7)]
print(f"Rows After Dropping Invalid ORI7: {len(lemas_clean):,}")

Rows Before Dropping Invalid ORI7: 3,499
Rows After Dropping Invalid ORI7: 3,288


#### 3.3 Analysis

In [22]:
# final check
print(f"Shape: {lemas_clean.shape[0]:,} rows, {lemas_clean.shape[1]} columns")

print(f"Unique Agencies: {lemas_clean['ORI7'].nunique():,}")

print(f"\nSummary Statistics:")
lemas_clean[['FTSWORN', 'PTSWORN', 'TOTFTEMP', 
             'OPBUDGET', 'DET_SWN','PERS_TRN_ACAD', 
             'PERS_TRN_INSVC', 'PERS_EDU_MIN']].describe().round(2)

Shape: 3,288 rows, 10 columns
Unique Agencies: 3,288

Summary Statistics:


,FTSWORN,PTSWORN,TOTFTEMP,OPBUDGET,DET_SWN,PERS_TRN_ACAD,PERS_TRN_INSVC,PERS_EDU_MIN
count,3265.00,2551.00,2566.00,2.524000e+03,3288.0,2521.00,2534.00,2566.00
mean,147.49,3.22,250.52,3.603427e+07,17.7,715.24,43.64,3.73
std,776.95,9.67,1233.80,1.685508e+08,101.4,286.49,70.13,0.69
min,0.00,0.00,0.00,0.000000e+00,0.0,0.00,0.00,1.00
25%,8.00,0.00,11.00,9.982938e+05,0.0,560.00,22.00,4.00
50%,25.00,0.00,40.00,4.387168e+06,1.0,700.00,35.00,4.00
75%,120.00,3.00,187.00,2.353773e+07,12.0,860.00,43.00,4.00
max,34810.00,256.00,49970.00,5.500000e+09,4593.0,6720.00,1380.00,5.00


In [23]:
lemas_out_path = os.path.join(processed_dir, "lemas_clean.csv")
lemas_clean.to_csv(lemas_out_path, index = False)
print(f"Saved Cleaned LEMAS data to {lemas_out_path}")

Saved Cleaned LEMAS data to /Users/mariamkhan/Desktop/Thesis/Data/Processed/lemas_clean.csv


### 4. ACS Data

#### 4.1 Initial Exploration

In [24]:
print("\nPoverty Sample:")
poverty_df.head(3)


Poverty Sample:


,GEO_ID,NAME,S1701_C01_001E,S1701_C01_001M,S1701_C01_002E,S1701_C01_002M,S1701_C01_003E,S1701_C01_003M,S1701_C01_004E,S1701_C01_004M,...,S1701_C03_058M,S1701_C03_059E,S1701_C03_059M,S1701_C03_060E,S1701_C03_060M,S1701_C03_061E,S1701_C03_061M,S1701_C03_062E,S1701_C03_062M,Unnamed: 374
0,0500000US01001,"Autauga County, Alabama",58731,138,13751,137,3373,142,10378,138,...,(X),3.2,2.5,48.4,13.2,32.1,5.0,10.6,1.8,NaN
1,0500000US01003,"Baldwin County, Alabama",236041,360,50219,344,11997,319,38222,194,...,(X),4.0,2.1,35.2,6.1,34.9,3.8,10.5,1.0,NaN
2,0500000US01005,"Barbour County, Alabama",21650,108,5127,47,1360,89,3767,98,...,(X),7.6,6.5,38.8,16.5,46.9,7.6,21.9,2.8,NaN


In [25]:
print("\nUnemployment Sample:")
unemployment_df.head(3)


Unemployment Sample:


,GEO_ID,NAME,B23025_001E,B23025_001M,B23025_002E,B23025_002M,B23025_003E,B23025_003M,B23025_004E,B23025_004M,B23025_005E,B23025_005M,B23025_006E,B23025_006M,B23025_007E,B23025_007M,Unnamed: 16
0,0500000US01001,"Autauga County, Alabama",47508,227,28020,890,27070,978,26382,966,688,234,950,273,19488,896,NaN
1,0500000US01003,"Baldwin County, Alabama",195048,457,113778,1904,113171,1913,109556,1998,3615,584,607,266,81270,1942,NaN
2,0500000US01005,"Barbour County, Alabama",20253,94,9085,428,9074,430,8556,440,518,192,11,17,11168,432,NaN


#### 4.2 Extracting County FIPS Codes

ACS files identify geography via `GEO_ID`, formatted like `0500000US01001` for county-level tables. The last 5 digits are the county FIPS code (2-digit state + 3-digit county) that MAP's county field will need to be mapped to (per the proposal's county-name-to-FIPS lookup table).

In [26]:
# extract five-digit county fips from geo_id 
poverty_df['FIPS'] = poverty_df['GEO_ID'].str[-5:].str.zfill(5)
unemployment_df['FIPS'] = unemployment_df['GEO_ID'].str[-5:].str.zfill(5)

# check
print(f"Poverty FIPS Sample: {poverty_df['FIPS'].head(5).tolist()}")
print(f"Poverty Unique Counties: {poverty_df['FIPS'].nunique():,}")
print(f"\nUnemployment FIPS Sample: {unemployment_df['FIPS'].head(5).tolist()}")
print(f"Unemployment Unique Counties: {unemployment_df['FIPS'].nunique():,}")

Poverty FIPS Sample: ['01001', '01003', '01005', '01007', '01009']
Poverty Unique Counties: 3,222

Unemployment FIPS Sample: ['01001', '01003', '01005', '01007', '01009']
Unemployment Unique Counties: 3,222


/var/folders/nr/2gzj735s0qv94cz6w4dtg11h0000gn/T/ipykernel_6161/1679199562.py:2: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  poverty_df['FIPS'] = poverty_df['GEO_ID'].str[-5:].str.zfill(5)


#### 4.3 Selecting and Cleaning ACS Variables

The ACS tables use specific estimate fields for the socioeconomic measures required in the analysis. Poverty is taken directly from the reported percentage below the poverty level. The unemployment rate is calculated from the civilian labour force and unemployed population counts in the B23025 table.

ACS sentinel values representing unavailable estimates are recoded as missing values before the variables are used in the analysis. https://www.census.gov/data/developers/data-sets/acs-1year/notes-on-acs-estimate-and-annotation-values.html

In [27]:
acs_missing_codes = [-666666666, -999999999]

In [28]:
# select and clean poverty rate 
poverty_col = 'S1701_C03_001E'

poverty_clean = poverty_df[
    ['FIPS', 'NAME', poverty_col]
    ].copy()

poverty_clean = poverty_clean.rename(
    columns = {poverty_col: 'PovertyRate'})

# convert poverty rate to numeric
poverty_clean['PovertyRate'] = pd.to_numeric(
    poverty_clean['PovertyRate'],
    errors = 'coerce')

# replace sentinel codes 
poverty_clean['PovertyRate'] = poverty_clean[
    'PovertyRate'].replace(acs_missing_codes, np.nan)

# checks
print(f"Poverty Rate Missing: "
      f"{poverty_clean['PovertyRate'].isnull().sum():,}")

print(poverty_clean['PovertyRate'].describe().round(2))


Poverty Rate Missing: 0
count    3222.00
mean       14.98
std         7.63
min         1.70
25%        10.10
50%        13.40
75%        17.80
max        64.70
Name: PovertyRate, dtype: float64


In [29]:
labor_force_col = 'B23025_003E' # civilian labor force
unemployed_col = 'B23025_005E' # civilian labor force: unemployed

unemployment_clean = unemployment_df[
    ['FIPS', 'NAME', labor_force_col, unemployed_col]
    ].copy()

unemployment_clean = unemployment_clean.rename(
    columns = {labor_force_col: 'CivilianLaborForce',
               unemployed_col: 'Unemployed'
})

# convert variables to numeric and replace sentinel codes 
for col in ['CivilianLaborForce', 'Unemployed']:
    unemployment_clean[col] = pd.to_numeric(unemployment_clean[col], errors = 'coerce')
    unemployment_clean[col] = unemployment_clean[col].replace(acs_missing_codes, np.nan)

# calculate the unemployment rate 
unemployment_clean['UnemploymentRate'] = (
    unemployment_clean['Unemployed'] / unemployment_clean['CivilianLaborForce'] * 100
)

# checks
print(f"Unemployment Rate Missing: "
      f"{unemployment_clean['UnemploymentRate'].isnull().sum():,}")

print(unemployment_clean['UnemploymentRate'].describe().round(2))

Unemployment Rate Missing: 0
count    3222.00
mean        4.92
std         2.78
min         0.00
25%         3.33
50%         4.53
75%         5.90
max        31.10
Name: UnemploymentRate, dtype: float64


#### 4.4 Combining ACS Data 

In [30]:
# combine poverty and unemployemt data using county fips
acs_clean = poverty_clean[
    ['FIPS', 'NAME', 'PovertyRate']].merge(
        unemployment_clean[['FIPS', 'UnemploymentRate']],
        on = 'FIPS', how = 'outer')

#### 4.5 Final Checks and Export

In [31]:
print(f"ACS Clean Shape: {acs_clean.shape}")
print(f"Unique Counties: {acs_clean['FIPS'].nunique():,}")
print(f"\nMissing PovertyRate: {acs_clean['PovertyRate'].isnull().sum():,}")
print(f"Missing UnemploymentRate: {acs_clean['UnemploymentRate'].isnull().sum():,}")
acs_clean.head()

ACS Clean Shape: (3222, 4)
Unique Counties: 3,222

Missing PovertyRate: 0
Missing UnemploymentRate: 0


,FIPS,NAME,PovertyRate,UnemploymentRate
0,01001,"Autauga County, Alabama",10.7,2.541559
1,01003,"Baldwin County, Alabama",10.5,3.194281
2,01005,"Barbour County, Alabama",21.9,5.708618
3,01007,"Bibb County, Alabama",20.5,9.984000
4,01009,"Blount County, Alabama",14.1,5.835173


In [32]:
# save the cleaned ACS dataset
acs_out_path = os.path.join(processed_dir, "acs_clean.csv")
acs_clean.to_csv(acs_out_path, index = False)
print(f"Saved Cleaned ACS Data to {acs_out_path}")

Saved Cleaned ACS Data to /Users/mariamkhan/Desktop/Thesis/Data/Processed/acs_clean.csv


### 5. County FIPS Crosswalk (MAP → ACS)

In [33]:
# snapshot row count before the crosswalk merges
# id is not a unique row identifier - multi-victim cases have the same id
n_rows_before_crosswalk = len(map_df)
n_rows_before_crosswalk

878210

#### 5.1. Deriving State from CNTYFIPS

MAP and ACS both describe counties, but in different formats:

- MAP says `"Anchorage, AK"`
- ACS says `"Anchorage Municipality, Alaska"` 

Before these two datasets can be joined, MAP's county names need to be translated into the same format ACS uses.

In [34]:
# map state abbreviations to full state names 
state_abbr_to_name = {
    'AL':'Alabama','AK':'Alaska','AZ':'Arizona','AR':'Arkansas','CA':'California',
    'CO':'Colorado','CT':'Connecticut','DE':'Delaware','DC':'District of Columbia',
    'FL':'Florida','GA':'Georgia','HI':'Hawaii','ID':'Idaho','IL':'Illinois',
    'IN':'Indiana','IA':'Iowa','KS':'Kansas','KY':'Kentucky','LA':'Louisiana',
    'ME':'Maine','MD':'Maryland','MA':'Massachusetts','MI':'Michigan','MN':'Minnesota',
    'MS':'Mississippi','MO':'Missouri','MT':'Montana','NE':'Nebraska','NV':'Nevada',
    'NH':'New Hampshire','NJ':'New Jersey','NM':'New Mexico','NY':'New York',
    'NC':'North Carolina','ND':'North Dakota','OH':'Ohio','OK':'Oklahoma','OR':'Oregon',
    'PA':'Pennsylvania','RI':'Rhode Island','SC':'South Carolina','SD':'South Dakota',
    'TN':'Tennessee','TX':'Texas','UT':'Utah','VT':'Vermont','VA':'Virginia',
    'WA':'Washington','WV':'West Virginia','WI':'Wisconsin','WY':'Wyoming'
}

def parse_cnty(cntyfips): 
    # split county name and state abbreviation
    parts = cntyfips.rsplit(',', 1)

    # convert the state abbr to full name 
    if len(parts) == 2 and parts[1].strip() in state_abbr_to_name:
        return parts[0].strip(), state_abbr_to_name[parts[1].strip()]

    # district of columbia doesn't have a state suffix 
    return cntyfips.strip(), 'District of Columbia'

# extract county name and state from cntyfips
parsed = map_df['CNTYFIPS'].apply(parse_cnty)

# store  in seperate columns 
map_df['county_name'] = parsed.apply(lambda x: x[0])
map_df['state_derived'] = parsed.apply(lambda x: x[1])

# compare against the existing State column to surface any mislabeled rows
mismatch = map_df[map_df['state_derived'] != map_df['State']]

print(f"State Mismatches: {len(mismatch):,}\n")

print(mismatch[['CNTYFIPS','State','state_derived']].drop_duplicates())


State Mismatches: 1,554

                    CNTYFIPS          State         state_derived
198911  District of Columbia     California  District of Columbia
727391           Bristol, RI  Rhodes Island          Rhode Island
727405              Kent, RI  Rhodes Island          Rhode Island
727519           Newport, RI  Rhodes Island          Rhode Island
727583        Providence, RI  Rhodes Island          Rhode Island
728863        Washington, RI  Rhodes Island          Rhode Island
901963              Dane, WI     Washington             Wisconsin


This turns up three issues in the raw MAP `State` column: 
- "Rhode Island" is misspelled `"Rhodes Island"` (~1,550 rows)
- One `Dane, WI` row is mislabeled `"Washington"`
- One `District of Columbia` row is mislabeled `"California"`.

We use `state_derived` from here on.

#### 5.2 Two-Pass Matching

County names are formatted differently in MAP and ACS, so a normalised county key is used to make them consistent. This handles differences such as `"De Kalb"` vs. `"DeKalb County"` and removes accents and common geographic suffixes. Because MAP is inconsistent in how it labels independent cities, matching is done in two steps. A strict match is attempted first, followed by a looser match for records that remain unmatched. 

In [35]:
def strip_accents(s):
    return ''.join(
        c for c in unicodedata.normalize('NFKD', s)
        if not unicodedata.combining(c)
    )

def normalize_key(name, strip_city = True):
    # remove accents and convert to lowercase
    name = strip_accents(name.lower()) 

    # remove suffixes 
    suffixes = (
        r'\b(county|parish|borough|census area|'
        r'municipality|municipio|city and borough'
    )
    suffixes += r'|city)\b' if strip_city else r')\b'
    name = re.sub(suffixes, '', name)

    # remove remaining non-letter characters
    name = re.sub(r'[^a-z]', '', name)

    return name

In [36]:
# build key columns 
acs_clean['county_key'] = acs_clean['NAME'].apply(
    lambda x: normalize_key(x.rsplit(',', 1)[0], strip_city = False)
)

acs_clean['county_key_loose'] = acs_clean['NAME'].apply(
    lambda x: normalize_key(x.rsplit(',', 1)[0], strip_city = True)
)

acs_clean['state_full'] = acs_clean['NAME'].apply(
    lambda x: x.rsplit(',', 1)[1].strip()
)

In [37]:
map_df['county_key'] = map_df['county_name'].apply(
    lambda x: normalize_key(x, strip_city = False)
)

map_df['county_key_loose'] = map_df['county_name'].apply(
    lambda x: normalize_key(x, strip_city = True)
)

In [38]:
# pass 1: strict match 
map_df = map_df.merge(
    acs_clean[['county_key', 'state_full', 'FIPS']],
    left_on = ['county_key', 'state_derived'],
    right_on = ['county_key', 'state_full'],
    how = 'left'
)

# pass 2: loose fallback for rows still unmatched 
acs_loose_pool = acs_clean[
    ~acs_clean.duplicated(
        subset = ['county_key_loose', 'state_full'],
        keep = False)
]

In [39]:
# find rows that didn't get a fips in pass 1
still_missing = map_df['FIPS'].isnull()

fallback = map_df.loc[
    still_missing,
    ['county_key_loose', 'state_derived']
].merge(
    acs_loose_pool[['county_key_loose', 'state_full', 'FIPS']],
    left_on = ['county_key_loose', 'state_derived'],
    right_on = ['county_key_loose', 'state_full'],
    how = 'left'
)
map_df.loc[still_missing, 'FIPS'] = fallback['FIPS'].values

# check success
match_rate = map_df['FIPS'].notnull().mean()

print(
    f"Matched Rows: "
    f"{map_df['FIPS'].notnull().sum():,} ({match_rate:.2%})"
)

print(
    f"Unmatched Rows: "
    f"{map_df['FIPS'].isnull().sum():,} ({1 - match_rate:.2%})"
)

Matched Rows: 871,797 (99.27%)
Unmatched Rows: 6,413 (0.73%)


In [40]:
unmatched = map_df[map_df['FIPS'].isnull()]

print("Unmatched County/State Combinations:")
print(unmatched[['CNTYFIPS', 'state_derived']]
      .drop_duplicates()
      .sort_values(['state_derived', 'CNTYFIPS'])
)

Unmatched County/State Combinations:
                                   CNTYFIPS state_derived
1335    Prince of Wales-Outer Ketchikan, AK        Alaska
1199              Skagway-Hoonah-Angoon, AK        Alaska
1206                     Valdez-Cordova, AK        Alaska
1188                Wrangell-Petersburg, AK        Alaska
185976                        Fairfield, CT   Connecticut
185964                         Hartford, CT   Connecticut
190464                       Litchfield, CT   Connecticut
187392                        Middlesex, CT   Connecticut
185940                        New Haven, CT   Connecticut
187738                       New London, CT   Connecticut
187399                          Tolland, CT   Connecticut
190721                          Windham, CT   Connecticut
721343                          Shannon, SD  South Dakota
839960                    Clifton Forge, VA      Virginia


#### 5.3 Virginia Overrides

Some areas in Maryland, Missouri and Virginia have independent cities that are separate from counties with the same name. These have different ACS entries and FIPS codes. For Baltimore and St. Louis, the MAP `CNTYFIPS` values distinguish the city from the county, so the original merge works correctly. 

For four Virginia locations, MAP does not clearly distinguish between the city and county. We will use the `Agency` field to identify the correct geography and correct the FIPS codes.

In [41]:
# define agency-based rules for va independent cities
va_city_county_overrides = {
    'Fairfax, VA': {'city_agency': ['Fairfax City'],
                    'city_fips': '51600', 
                    'county_fips': '51059'},

    'Franklin, VA': {'city_agency': ['Franklin'],
                    'city_fips': '51620', 
                    'county_fips': '51067'},

    'Richmond, VA': {'city_agency': ['Richmond', 'State Police: Richmond'],
                    'city_fips': '51760', 
                    'county_fips': '51159'},

    'Roanoke, VA':{'city_agency': ['Roanoke', 'State Police: Roanoke'],
                    'city_fips': '51770', 
                    'county_fips': '51161'}
}

In [42]:
# apply the city fips code 
for cntyfips, rule in va_city_county_overrides.items():

    # find records belonging to the ind city 
    is_city = (
        (map_df['CNTYFIPS'] == cntyfips) 
        & (map_df['Agency'].isin(rule['city_agency']))
    )

    # count how many are being corrected 
    n_corrected = is_city.sum()

    # replace county fips with city fips
    map_df.loc[is_city, 'FIPS'] = rule['city_fips']

    print(
        f"{cntyfips}: {n_corrected:,} rows corrected"
        f" to city FIPS {rule['city_fips']}"
    )

Fairfax, VA: 15 rows corrected to city FIPS 51600
Franklin, VA: 50 rows corrected to city FIPS 51620
Richmond, VA: 3,672 rows corrected to city FIPS 51760
Roanoke, VA: 627 rows corrected to city FIPS 51770


#### 5.4 Manual Overrides for County Changes

- Shannon County was officially renamed Oglala Lakota County (46102) in May 2015
- Clifton Forge reverted to a town incorporated within Alleghany County (51005) in July 2001

In [43]:
# create dictionary
manual_fips_overrides = {
    ('Shannon, SD', 'South Dakota'): '46102',
    ('Clifton Forge, VA', 'Virginia'): '51005'
}

for (cntyfips, state), fips in manual_fips_overrides.items():

    # identify rows matching historical county and state 
    mask = (map_df['CNTYFIPS'] == cntyfips) & (map_df['state_derived'] == state)

    # assign current acs fips code 
    map_df.loc[mask, 'FIPS'] = fips

# recalculate the match rate
match_rate = map_df['FIPS'].notnull().mean()

print(
    f"Matched Rows (After Override): "
    f"{map_df['FIPS'].notnull().sum():,} ({match_rate:.2%})"
)

Matched Rows (After Override): 871,833 (99.27%)


A small number of MAP records remain unmatched to an ACS FIPS code because some county boundaries have changed over time. 

- **Connecticut:** The ACS uses 9 counties instead of the original 8. 
- **Alaska:** Four historical boroughs have been split into multiple areas. 

Without more detailed location information, these records can't be matched reliably, and will have missing `PovertyRate` and `UnemploymentRate` values rather than being assigned uncertain FIPS codes.

#### 5.5 Final Clean and Export 


In [44]:
# overwrite the state column with the corrected state_derived
n_fixed = (map_df['state_derived'] != map_df['State']).sum()
print(f"Overwriting {n_fixed:,} Rows")

map_df['State'] = map_df['state_derived']
print("Unique States After Overwrite:")
print(map_df['State'].unique())

Overwriting 1,554 Rows
Unique States After Overwrite:
<StringArray>
[              'Alaska',              'Alabama',             'Arkansas',
              'Arizona',           'California',             'Colorado',
          'Connecticut', 'District of Columbia',             'Delaware',
              'Florida',              'Georgia',               'Hawaii',
                 'Iowa',                'Idaho',             'Illinois',
              'Indiana',               'Kansas',             'Kentucky',
            'Louisiana',        'Massachusetts',             'Maryland',
                'Maine',             'Michigan',            'Minnesota',
             'Missouri',          'Mississippi',              'Montana',
             'Nebraska',       'North Carolina',         'North Dakota',
        'New Hampshire',           'New Jersey',           'New Mexico',
               'Nevada',             'New York',                 'Ohio',
             'Oklahoma',               'Oregon',        

In [45]:
# store fips as a five digit string 
map_df['FIPS'] = map_df['FIPS'].apply(
    lambda x: str(x).zfill(5) if pd.notnull(x) else np.nan
)

# remove temporary columns 
map_df = map_df.drop(
    columns = ['county_name', 'state_derived', 'county_key_loose', 
               'county_key', 'state_full'],
    errors = 'ignore')

# save cleaned map with the resolved fips column
map_out_path = os.path.join(processed_dir, "map_clean.csv")
map_df.to_csv(map_out_path, index = False)

print(f"Resaved Clean MAP with FIPS to {map_out_path}")
print(f"Shape: {map_df.shape}")

Resaved Clean MAP with FIPS to /Users/mariamkhan/Desktop/Thesis/Data/Processed/map_clean.csv
Shape: (878210, 22)


In [46]:
# regression check: confirm the merges above didn't duplicate any rows
print("Rows Before Crosswalk Merges:", n_rows_before_crosswalk)
print("Rows After Crosswalk Merges:", len(map_df))

Rows Before Crosswalk Merges: 878210
Rows After Crosswalk Merges: 878210


### 6. Final Fused Dataset (MAP + LEMAS + ACS)

#### 6.1 Pre-Join Checks

In [47]:
print(f"Duplicate ORI7 in LEMAS data: {lemas_clean['ORI7'].duplicated().sum():,}")
print(f"Duplicate FIPS in ACS data: {acs_clean['FIPS'].duplicated().sum():,}")

Duplicate ORI7 in LEMAS data: 0
Duplicate FIPS in ACS data: 0


In [48]:
n_rows_before_join = len(map_df)
print(f"Rows Before Joining LEMAS and ACS: {n_rows_before_join:,}")

Rows Before Joining LEMAS and ACS: 878,210


#### 6.2 Join LEMAS and ACS onto MAP

- `lemas_clean` and `map_df` via `Ori/ORI7`
- `acs_clean` via `FIPS`

In [49]:
fused_df = map_df.merge(
    lemas_clean, 
    left_on = 'Ori', 
    right_on = 'ORI7', 
    how = 'left', 
    indicator = '_lemas_merge'
).merge(
    acs_clean[['FIPS', 'NAME', 'PovertyRate', 'UnemploymentRate']], 
    on = 'FIPS', 
    how = 'left',
    indicator = '_acs_merge'
)

print(f"Rows After Joining LEMAS and ACS: {len(fused_df):,}")

Rows After Joining LEMAS and ACS: 878,210


#### 6.3 Coverage Check

Not every MAP row will find a LEMAS or ACS match. This quantifies exactly how much of the fused dataset has each source available, which matters for RQ2. 

In [50]:
lemas_matched = (fused_df['_lemas_merge'] == 'both').sum()
acs_matched = (fused_df['_acs_merge'] == 'both').sum()

lemas_percent = lemas_matched / len(fused_df) * 100
acs_percent = acs_matched / len(fused_df) * 100

print(f"LEMAS Matched: {lemas_matched:,} ({lemas_percent:.2f}%)")
print(f"ACS Matched: {acs_matched:,} ({acs_percent:.2f}%)")

LEMAS Matched: 695,643 (79.21%)
ACS Matched: 871,833 (99.27%)


#### 6.4 Comparing Cases With and Without LEMAS Data

LEMAS does not cover all police agencies equally, so cases with LEMAS data might differ from those without it. We compare solved rates and victim race distributions between the two groups to check whether there are noticeable differences that could affect the analysis. 

In [51]:
lemas_has_data = fused_df['_lemas_merge'] == 'both'

print(f"Solved Rate (With vs Without LEMAS Data):")
print(fused_df.groupby(lemas_has_data)['SOLVED'].mean().round(3))

print("\nVictim Race Distribution (With vs Without LEMAS Data):")
print(pd.crosstab(fused_df['VicRace'], 
                  lemas_has_data, 
                  margins = True, 
                  normalize = 'columns')
                  .round(3)
)


Solved Rate (With vs Without LEMAS Data):
_lemas_merge
False    0.811
True     0.678
Name: SOLVED, dtype: float64

Victim Race Distribution (With vs Without LEMAS Data):
_lemas_merge                         False   True    All
VicRace                                                 
American Indian or Alaskan Native    0.016  0.006  0.008
Asian                                0.012  0.015  0.014
Black                                0.319  0.528  0.485
Native Hawaiian or Pacific Islander  0.000  0.000  0.000
Unknown                              0.010  0.010  0.010
White                                0.643  0.440  0.482


Cases with LEMAS data have a lower solved rate (67.8%) than cases without LEMAS data (81.1%). The racial composition also differs. The differences suggest that LEMAS coverage is not evenly distributed across the MAP cases and should be considered when interpreting analyses that use the LEMAS variables.

In [52]:
lemas_cols = ['FTSWORN', 'PTSWORN', 'TOTFTEMP', 
              'OPBUDGET', 'DET_SWN', 'PERS_TRN_ACAD', 
              'PERS_TRN_INSVC', 'PERS_EDU_MIN']

print("Missing Rates (All Cases):")
print((fused_df[lemas_cols].isnull().mean() * 100).round(2))

print("\nMissing Rate (LEMAS Matched Cases):")
print((fused_df.loc[lemas_has_data, lemas_cols].isnull().mean() * 100).round(2))

Missing Rates (All Cases):
FTSWORN           20.79
PTSWORN           26.99
TOTFTEMP          26.88
OPBUDGET          27.24
DET_SWN           20.79
PERS_TRN_ACAD     27.25
PERS_TRN_INSVC    27.22
PERS_EDU_MIN      26.85
dtype: float64

Missing Rate (LEMAS Matched Cases):
FTSWORN           0.00
PTSWORN           7.83
TOTFTEMP          7.69
OPBUDGET          8.14
DET_SWN           0.00
PERS_TRN_ACAD     8.16
PERS_TRN_INSVC    8.12
PERS_EDU_MIN      7.66
dtype: float64


#### 6.5 Save

In [53]:
fused_out_path = os.path.join(processed_dir, "fused_clean.csv")
fused_df.to_csv(fused_out_path, index = False)

print(f"Saved Fused Cleaned Data to {fused_out_path}")
print(f"Shape: {fused_df.shape}")
print(f"\nColumns in Fused Dataset: {list(fused_df.columns)}")


Saved Fused Cleaned Data to /Users/mariamkhan/Desktop/Thesis/Data/Processed/fused_clean.csv
Shape: (878210, 37)

Columns in Fused Dataset: ['ID', 'CNTYFIPS', 'Ori', 'State', 'Agency', 'Agentype', 'Year', 'Month', 'Incident', 'VicAge', 'VicSex', 'VicRace', 'Weapon', 'Circumstance', 'VicCount', 'OffCount', 'SOLVED', 'VicAgeBand', 'agency_year_month', 'Monthly_Overlap', 'NearLemas', 'FIPS', 'ORI7', 'STATE', 'FTSWORN', 'PTSWORN', 'TOTFTEMP', 'OPBUDGET', 'DET_SWN', 'PERS_TRN_ACAD', 'PERS_TRN_INSVC', 'PERS_EDU_MIN', '_lemas_merge', 'NAME', 'PovertyRate', 'UnemploymentRate', '_acs_merge']
